<a href="https://colab.research.google.com/github/rayynaldgitau/ArtSpace/blob/master/Hatespeech_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import ast
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
from google.colab import drive

drive.mount('/content/drive')


df = pd.read_csv('/content/drive/MyDrive/USIU/HateSpeech_Kenya.csv')
df.head()

RANDOM_STATE = 42
TEST_SIZE = 0.2

CLASS_MAP = {0: "Neither", 1: "Offensive", 2: "Hate Speech"}
LABEL_ORDER = ["Hate Speech", "Offensive", "Neither"]


def extract_tweet(raw):
    parsed = ast.literal_eval(raw)
    return parsed[0] if len(parsed) > 0 else ""


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"username_\d+", " " , text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_and_prepare(path):
    df = pd.read_csv(path)
    df["raw_text"] = df["Tweet"].apply(extract_tweet)
    df["label"] = df["Class"].map(CLASS_MAP)
    df["clean_text"] = df["raw_text"].apply(clean_text)
    df = df[df["clean_text"].str.len() > 0]
    print(f"Loaded {len(df)} rows after cleaning.")
    print("\nClass distribution:")
    print(df["label"].value_counts())
    print("\nClass distribution (%):")
    print((df["label"].value_counts(normalize=True) * 100).round(1))
    return df


def train_and_evaluate(df):
    X_train, X_test, y_train, y_test = train_test_split(
        df["clean_text"], df["label"],
        test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label"]
    )

    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    models = {
        "Naive Bayes": MultinomialNB(),
        "SVM (Linear)": LinearSVC(random_state=RANDOM_STATE, max_iter=5000, class_weight="balanced"),
        "Logistic Regression": LogisticRegression(
            random_state=RANDOM_STATE, max_iter=2000, class_weight="balanced"
        ),
    }

    results = []
    predictions = {}
    reports = {}

    for name, model in models.items():
        model.fit(X_train_vec, y_train)
        y_pred = model.predict(X_test_vec)
        predictions[name] = y_pred

        macro_f1 = f1_score(y_test, y_pred, average="macro")
        weighted_f1 = f1_score(y_test, y_pred, average="weighted")
        per_class = f1_score(y_test, y_pred, average=None, labels=LABEL_ORDER)
        results.append({
            "Model": name,
            "Macro F1": macro_f1,
            "Weighted F1": weighted_f1,
            "F1 (Hate Speech)": per_class[0],
            "F1 (Offensive)": per_class[1],
            "F1 (Neither)": per_class[2],
        })
        reports[name] = classification_report(y_test, y_pred, digits=3)

        print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
        print(reports[name])

    results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
    print("\nFINAL COMPARISON (sorted by Macro F1)")
    print(results_df.to_string(index=False))
    results_df.to_csv("/content/drive/MyDrive/f1_comparison.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(results_df))
    width = 0.35
    ax.bar(x - width / 2, results_df["Macro F1"], width, label="Macro F1")
    ax.bar(x + width / 2, results_df["Weighted F1"], width, label="Weighted F1")
    ax.set_xticks(x)
    ax.set_xticklabels(results_df["Model"])
    ax.set_ylabel("F1 Score")
    ax.set_title("Model Comparison: F1 Score by Algorithm (Hate Speech, Offensive, Neither)")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/f1_comparison.png", dpi=150)
    plt.close()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (name, y_pred) in zip(axes, predictions.items()):
        cm = confusion_matrix(y_test, y_pred, labels=LABEL_ORDER)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_ORDER)
        disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
        ax.set_title(name)
    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/confusion_matrices.png", dpi=150) # Changed path
    plt.close()

    return results_df, reports


if __name__ == "__main__":
    data_path = '/content/drive/MyDrive/USIU/HateSpeech_Kenya.csv'
    processed_df = load_and_prepare(data_path)
    train_and_evaluate(processed_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 48059 rows after cleaning.

Class distribution:
label
Neither        36335
Offensive       8543
Hate Speech     3181
Name: count, dtype: int64

Class distribution (%):
label
Neither        75.6
Offensive      17.8
Hate Speech     6.6
Name: proportion, dtype: float64

Naive Bayes
              precision    recall  f1-score   support

 Hate Speech      0.667     0.035     0.066       636
     Neither      0.764     0.994     0.864      7267
   Offensive      0.458     0.035     0.065      1709

    accuracy                          0.760      9612
   macro avg      0.630     0.354     0.332      9612
weighted avg      0.703     0.760     0.669      9612


SVM (Linear)
              precision    recall  f1-score   support

 Hate Speech      0.293     0.398     0.337       636
     Neither      0.837     0.816     0.827      7267
   Offensive      0.332   